### Checks

`Urban Ward_Panel` sheets.


1. All TV_CODES should show up at least once. Either one shape for a town or village (NaN Ward code) or multiple ward shapes with codes.
2. Which TV_Codes have we been given multiple shapes for (wards)? Should match or overdeliver on their promise.

Add column that says "Delivery State": 
- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

## Setup

In [2]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [3]:
# general
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm

# for plotting and coloring
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import math
import matplotlib.cm

gpd.options.io_engine = "pyogrio"

In [4]:
from gridsample.utils import create_ids, create_gmap_links, save_shapefiles
from gridsample.mapping import create_interactive_map

In [5]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"
CLEANED_DATA_DIR = (
    DATA_DIR / "Panel" / "02. Intermediate Outputs" / "00. Merge and Quality Checks" / "v1"
)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
def generate_colormap(N):
    arr = np.arange(N) / N
    N_up = int(math.ceil(N / 7) * 7)
    arr.resize(N_up)
    arr = arr.reshape(7, N_up // 7).T.reshape(-1)
    ret = matplotlib.cm.hsv(arr)
    n = ret[:, 3].size
    a = n // 2
    b = n - a

    # Create arrays of exactly matching sizes
    for i in range(3):
        ret[0:a, i] *= np.linspace(0.2, 1, a)
    ret[a:, 3] *= np.linspace(1, 0.3, b)

    return ret[:N]  # Return only the requested number of colors

## 0. Load request excel

In [7]:
excel_file = (
    RAW_DATA_DIR / "00. Boundary Requests" / "SamplingOutput_Summary_IFS & Panel.xlsx"
)

In [8]:
panel_df = pd.read_excel(excel_file, sheet_name="Urban Ward_Panel")

In [9]:
panel_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Included in Panel,Selected in IFS Sample,Ward Boundary Available with MapSolve
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,16,Shimla (M Corp.) WARD NO.-0016,Urban,5196,"1,69,578","68,64,602",033-00182-800137-0016,No,No,No
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,22,Shimla (M Corp.) WARD NO.-0022,Urban,15399,"1,69,578","68,64,602",033-00182-800137-0022,No,No,No
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,23,Shimla (M Corp.) WARD NO.-0023,Urban,12029,"1,69,578","68,64,602",033-00182-800137-0023,No,No,No
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800165,7,Jalandhar Cantt. (CB) WARD NO.-0007,Urban,6349,"11,45,692","2,77,43,338",037-00212-800165-0007,No,No,Yes
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,1,Jalandhar (M Corp.) WARD NO.-0001,Urban,27750,"11,45,692","2,77,43,338",037-00212-800166-0001,No,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574163,0,Korremul,Rural,9922,"1,88,380",NaN,537-04523-574163-0000,Yes,No,No
708,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574173,1,Peerzadguda (CT) WARD NO.-0001,Urban,32586,"1,88,380",NaN,537-04523-574173-0001,Yes,No,No
709,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,4,Mahbubnagar (M) WARD NO.-0004,Urban,12175,"2,49,091",NaN,538-04568-802922-0004,Yes,Yes,No
710,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579685,1,Khanapuram Haveli (CT) WARD NO.-0001,Urban,53442,"3,13,504",NaN,541-04757-579685-0001,Yes,No,No


In [10]:
# sort and reset index
panel_df = panel_df.sort_values(
    by=["State", "District", "Subdistrict", "TownVillage", "UrbanWardVillage"]
).reset_index(drop=True)

In [11]:
panel_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Included in Panel,Selected in IFS Sample,Ward Boundary Available with MapSolve
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,16,Shimla (M Corp.) WARD NO.-0016,Urban,5196,"1,69,578","68,64,602",033-00182-800137-0016,No,No,No
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,22,Shimla (M Corp.) WARD NO.-0022,Urban,15399,"1,69,578","68,64,602",033-00182-800137-0022,No,No,No
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,23,Shimla (M Corp.) WARD NO.-0023,Urban,12029,"1,69,578","68,64,602",033-00182-800137-0023,No,No,No
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800165,7,Jalandhar Cantt. (CB) WARD NO.-0007,Urban,6349,"11,45,692","2,77,43,338",037-00212-800165-0007,No,No,Yes
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,1,Jalandhar (M Corp.) WARD NO.-0001,Urban,27750,"11,45,692","2,77,43,338",037-00212-800166-0001,No,No,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574163,0,Korremul,Rural,9922,"1,88,380",NaN,537-04523-574163-0000,Yes,No,No
708,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574173,1,Peerzadguda (CT) WARD NO.-0001,Urban,32586,"1,88,380",NaN,537-04523-574173-0001,Yes,No,No
709,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,4,Mahbubnagar (M) WARD NO.-0004,Urban,12175,"2,49,091",NaN,538-04568-802922-0004,Yes,Yes,No
710,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579685,1,Khanapuram Haveli (CT) WARD NO.-0001,Urban,53442,"3,13,504",NaN,541-04757-579685-0001,Yes,No,No


In [12]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards.csv", index=False)

In [13]:
# get unique across district and subdistrict both
len(panel_df[["State", "District", "Subdistrict"]].drop_duplicates())

247

#### Check for duplicated wards in our own sample

In [14]:
duplicated_sampled_wards = panel_df[
    panel_df[["TownVillage", "UrbanWardVillage"]].duplicated(keep=False)
].sort_values(["TownVillage", "UrbanWardVillage"])
duplicated_sampled_wards

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Included in Panel,Selected in IFS Sample,Ward Boundary Available with MapSolve
103,7,NCT OF DELHI,90,North West,431,Saraswati Vihar,800441,59,DMC (U) (Part) WARD NO.-0059,Urban,45964,"22,50,816","1,67,87,941",090-00431-800441-0059,No,No,Yes
154,7,NCT OF DELHI,96,West,450,Punjabi Bagh,800441,59,DMC (U) (Part) WARD NO.-0059,Urban,10377,"7,99,453","1,67,87,941",096-00450-800441-0059,No,Yes,Yes
142,7,NCT OF DELHI,96,West,448,Patel Nagar,800441,99,DMC (U) (Part) WARD NO.-0099,Urban,35330,"12,62,158","1,67,87,941",096-00448-800441-0099,No,No,Yes
155,7,NCT OF DELHI,96,West,450,Punjabi Bagh,800441,99,DMC (U) (Part) WARD NO.-0099,Urban,16525,"7,99,453","1,67,87,941",096-00450-800441-0099,No,No,Yes


In [15]:
duplicated_sampled_wards.to_csv(
    CLEANED_DATA_DIR / "Duplicated Sampled Wards.csv", index=False
)

## 1. Load all boundaries

In [16]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]

In [17]:
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True)

In [18]:
gdf = gdf.drop_duplicates()

#### Fix Chittoor issue

In [19]:
gdf[gdf["Dist_N"] == "Chittoor"]

,UID,PCA_ID,State_C,State_N,Dist_C,Dist_N,SubDist_C,SubDist_N,TV_C,TV_N,Ward_C,TOT_P,geometry
7583,1001873,803014,28.0,Andhra Pradesh,554.0,Chittoor,0.0,None,803014.0,Tirupati (M Corp.),NaN,287482.0,"MULTIPOLYGON (((8841030.000 1534620.000, 88410..."
7656,1015953,596011,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596011.0,Tirupati (NMA) (CT),NaN,37968.0,"MULTIPOLYGON (((8848740.000 1533150.000, 88487..."
7657,1001983,596010,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596010.0,Tirumala (CT),NaN,7741.0,"MULTIPOLYGON (((8832750.000 1540950.000, 88328..."
7658,1001886,596013,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596013.0,Mangalam (CT),NaN,19318.0,"MULTIPOLYGON (((8845260.000 1535520.000, 88453..."
7659,1001877,596012,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596012.0,Akkarampalle (CT),NaN,44219.0,"MULTIPOLYGON (((8842290.000 1535010.000, 88423..."
7660,1001889,596009,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596009.0,Chennayyagunta,NaN,3984.0,"MULTIPOLYGON (((8848110.000 1535730.000, 88484..."
7661,1001864,596008,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596008.0,Konkachennaiahgunta,NaN,581.0,"MULTIPOLYGON (((8848140.000 1534140.000, 88482..."


In [20]:
gdf.loc[gdf["TV_C"] == 803014, "SubDist_N"] = "Tirupati (Urban)"
gdf.loc[gdf["TV_C"] == 803014, "SubDist_C"] = 5383

In [21]:
gdf[gdf["Dist_N"] == "Chittoor"]

,UID,PCA_ID,State_C,State_N,Dist_C,Dist_N,SubDist_C,SubDist_N,TV_C,TV_N,Ward_C,TOT_P,geometry
7583,1001873,803014,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),803014.0,Tirupati (M Corp.),NaN,287482.0,"MULTIPOLYGON (((8841030.000 1534620.000, 88410..."
7656,1015953,596011,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596011.0,Tirupati (NMA) (CT),NaN,37968.0,"MULTIPOLYGON (((8848740.000 1533150.000, 88487..."
7657,1001983,596010,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596010.0,Tirumala (CT),NaN,7741.0,"MULTIPOLYGON (((8832750.000 1540950.000, 88328..."
7658,1001886,596013,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596013.0,Mangalam (CT),NaN,19318.0,"MULTIPOLYGON (((8845260.000 1535520.000, 88453..."
7659,1001877,596012,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596012.0,Akkarampalle (CT),NaN,44219.0,"MULTIPOLYGON (((8842290.000 1535010.000, 88423..."
7660,1001889,596009,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596009.0,Chennayyagunta,NaN,3984.0,"MULTIPOLYGON (((8848110.000 1535730.000, 88484..."
7661,1001864,596008,28.0,Andhra Pradesh,554.0,Chittoor,5383.0,Tirupati (Urban),596008.0,Konkachennaiahgunta,NaN,581.0,"MULTIPOLYGON (((8848140.000 1534140.000, 88482..."


## 2. Checks

### Any MapSolve rows with missing TV code?

In [22]:
gdf_no_TV_code_filtered = gdf[gdf["TV_C"].isna()]
gdf_no_TV_code_filtered

,UID,PCA_ID,State_C,State_N,Dist_C,Dist_N,SubDist_C,SubDist_N,TV_C,TV_N,Ward_C,TOT_P,geometry
24,16,None,22.0,Chhattisgarh,409.0,Durg,3317.0,Durg,NaN,None,NaN,NaN,"MULTIPOLYGON (((9037170.000 2420040.000, 90371..."
25,16,None,22.0,Chhattisgarh,406.0,Bilaspur,3295.0,Bilaspur,NaN,None,NaN,NaN,"MULTIPOLYGON (((9166590.000 2550750.000, 91665..."
1460,77,None,23.0,Madhya Pradesh,451.0,Jabalpur,3631.0,Jabalpur,NaN,None,NaN,NaN,"POLYGON ((8900430.000 2632320.000, 8900490.000..."
1658,9504321,55658,5.0,Uttarakhand,67.0,Udham Singh Nagar,346.0,Kashipur,NaN,None,NaN,283136.0,"MULTIPOLYGON (((8796240.000 3417120.000, 87963..."
1659,9488658,45145,5.0,Uttarakhand,60.0,Dehradun,304.0,Dehradun,NaN,None,NaN,988007.0,"MULTIPOLYGON (((8678280.000 3566610.000, 86783..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47977,240,7-95-447,7.0,NCT of Delhi,95.0,Central,447.0,Karol Bagh,NaN,NaN,NaN,136599.0,"MULTIPOLYGON (((8591760.000 3331200.000, 85918..."
47978,219,7-98-454,7.0,NCT of Delhi,98.0,South,454.0,Hauz Khas (Saket),NaN,NaN,NaN,1219100.0,"MULTIPOLYGON (((8595720.000 3300750.000, 85957..."
47979,155,7-97-453,7.0,NCT of Delhi,97.0,South West,453.0,Vasant Vihar,NaN,NaN,NaN,641666.0,"MULTIPOLYGON (((8586120.000 3312960.000, 85861..."
47980,156,7-97-452,7.0,NCT of Delhi,97.0,South West,452.0,Delhi Cantonment,NaN,NaN,NaN,286140.0,"MULTIPOLYGON (((8589120.000 3321540.000, 85891..."


In [23]:
gdf_no_TV_code_filtered.to_csv(
    CLEANED_DATA_DIR / "MapSolve Data without TV Codes.csv", index=False
)

### Request satisfaction check

In [24]:
given_states_list = list(gdf["State_C"].unique())
given_states_list.append(
    90
)  # manually add 90 for Telangana / Andhra Pradesh discrepency
len(given_states_list)

24

In [25]:
panel_df.loc[panel_df["State"].isin(given_states_list), "State_Name"].unique()

array(['HIMACHAL PRADESH', 'PUNJAB', 'UTTARAKHAND', 'HARYANA',
       'NCT OF DELHI', 'RAJASTHAN', 'UTTAR PRADESH', 'BIHAR', 'TRIPURA',
       'ASSAM', 'WEST BENGAL', 'JHARKHAND', 'ODISHA', 'CHHATTISGARH',
       'MADHYA PRADESH', 'GUJARAT', 'MAHARASHTRA', 'ANDHRA PRADESH',
       'KARNATAKA', 'KERALA', 'TAMIL NADU', 'TELANGANA'], dtype=object)

In [26]:
panel_df["State Shared by MapSolve"] = False
panel_df.loc[
    panel_df["State"].isin(given_states_list), "State Shared by MapSolve"
] = True

In [27]:
len(panel_df[~panel_df["State Shared by MapSolve"]])

0

In [28]:
panel_df["State Changed"] = "No"
panel_df.loc[panel_df["State_Name"] == "TELANGANA", "State Changed"] = (
    "Previously Andhra Pradesh"
)

In [29]:
panel_df

,State,State_Name,District,District_Name,Subdistrict,Subd_Name,TownVillage,UrbanWardVillage,WardVillage_Name,TRU,WardVillage_Pop,Subd_Pop,State_Pop,WardVillageID,Included in Panel,Selected in IFS Sample,Ward Boundary Available with MapSolve,State Shared by MapSolve,State Changed
0,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,16,Shimla (M Corp.) WARD NO.-0016,Urban,5196,"1,69,578","68,64,602",033-00182-800137-0016,No,No,No,True,No
1,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,22,Shimla (M Corp.) WARD NO.-0022,Urban,15399,"1,69,578","68,64,602",033-00182-800137-0022,No,No,No,True,No
2,2,HIMACHAL PRADESH,33,Shimla,182,Shimla (urban),800137,23,Shimla (M Corp.) WARD NO.-0023,Urban,12029,"1,69,578","68,64,602",033-00182-800137-0023,No,No,No,True,No
3,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800165,7,Jalandhar Cantt. (CB) WARD NO.-0007,Urban,6349,"11,45,692","2,77,43,338",037-00212-800165-0007,No,No,Yes,True,No
4,3,PUNJAB,37,Jalandhar,212,Jalandhar - I,800166,1,Jalandhar (M Corp.) WARD NO.-0001,Urban,27750,"11,45,692","2,77,43,338",037-00212-800166-0001,No,No,Yes,True,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
707,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574163,0,Korremul,Rural,9922,"1,88,380",NaN,537-04523-574163-0000,Yes,No,No,True,Previously Andhra Pradesh
708,90,TELANGANA,537,Rangareddy,4523,Ghatkesar,574173,1,Peerzadguda (CT) WARD NO.-0001,Urban,32586,"1,88,380",NaN,537-04523-574173-0001,Yes,No,No,True,Previously Andhra Pradesh
709,90,TELANGANA,538,Mahbubnagar,4568,Mahbubnagar,802922,4,Mahbubnagar (M) WARD NO.-0004,Urban,12175,"2,49,091",NaN,538-04568-802922-0004,Yes,Yes,No,True,Previously Andhra Pradesh
710,90,TELANGANA,541,Khammam,4757,Khammam (Urban),579685,1,Khanapuram Haveli (CT) WARD NO.-0001,Urban,53442,"3,13,504",NaN,541-04757-579685-0001,Yes,No,No,True,Previously Andhra Pradesh


#### Check for wards

In [30]:
panel_df["PCA_ID"] = (
    panel_df["TownVillage"].astype(str)
    + "-"
    + panel_df["UrbanWardVillage"].astype(str)
)
given_ward_ids = gdf["PCA_ID"].unique()

panel_df["Ward Boundary Given"] = False
panel_df.loc[panel_df["PCA_ID"].isin(given_ward_ids), "Ward Boundary Given"] = True

In [31]:
len(set(panel_df["PCA_ID"]).intersection(given_ward_ids))

319

#### Check for TownVillage codes

In [32]:
given_TV_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].notna(),
        "TV_C",
    ].astype(int)
)

panel_df["TV Boundary Given"] = False
panel_df.loc[
    panel_df["TownVillage"].astype(int).isin(given_TV_ids),
    "TV Boundary Given",
] = True

#### Check for SubDistricts

In [33]:
given_subdist_ids = set(
    gdf.loc[
        gdf["Ward_C"].isna() & gdf["TV_C"].isna() & gdf["SubDist_C"].notna(),
        "SubDist_C",
    ].astype(int)
)

panel_df["SubDistrict Boundary Given"] = False
panel_df.loc[
    panel_df["Subdistrict"].astype(int).isin(given_subdist_ids),
    "SubDistrict Boundary Given",
] = True

#### Fill in Delivery State

- BETTER - Ward boundary given but only TV or Subdistrict was expected
- GOOD - Ward boundary given as expected
- GOOD - Town/Village boundary given as expected
- GOOD - Subdistrict boundary given as expected
- BAD - Town/Village boundary given but Ward boundaries expected
- BAD - Subdistrict boundary given but Ward boundaries expected
- BAD - No boundary(s) given

In [34]:
## baseline
panel_df["Delivery State"] = "BAD - No boundary(s) given"

## subdistrict
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["SubDistrict Boundary Given"],
    "Delivery State",
] = "BAD - Subdistrict boundary given but Ward boundaries expected"

# tripura > dukli special case
panel_df.loc[
    (panel_df["State_Name"] == "TRIPURA") & (panel_df["Subd_Name"] == "Dukli"),
    "Delivery State",
] = "GOOD - Subdistrict boundary given as expected"

## town/village
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "GOOD - Town/Village boundary given as expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["TV Boundary Given"],
    "Delivery State",
] = "BAD - Town/Village boundary given but Ward boundaries expected"

## ward
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "No")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "BETTER - Ward boundary given but only TV or Subdistrict was expected"
panel_df.loc[
    (panel_df["Ward Boundary Available with MapSolve"] == "Yes")
    & panel_df["Ward Boundary Given"],
    "Delivery State",
] = "GOOD - Ward boundary given as expected"

In [35]:
panel_df["Delivery State"].value_counts()

Delivery State
GOOD - Town/Village boundary given as expected                          324
GOOD - Ward boundary given as expected                                  314
GOOD - Subdistrict boundary given as expected                            64
BETTER - Ward boundary given but only TV or Subdistrict was expected      7
BAD - No boundary(s) given                                                2
BAD - Town/Village boundary given but Ward boundaries expected            1
Name: count, dtype: int64

#### Add `PSU Type` column

In [36]:
psu_mapping = {
    "BETTER - Ward boundary given but only TV or Subdistrict was expected": "ward",
    "GOOD - Town/Village boundary given as expected": "town_village",
    "GOOD - Ward boundary given as expected": "ward",
    "GOOD - Subdistrict boundary given as expected": "subdistrict",
    "BAD - Town/Village boundary given but Ward boundaries expected": "town_village",
    "BAD - No boundary(s) given": "none",
    "BAD - Subdistrict boundary given but Ward boundaries expected": "subdistrict",
}
# create a new column in panel_df called PSU Type
panel_df["PSU Type"] = panel_df["Delivery State"].map(psu_mapping)

#### Add `Ward Count` column

In [37]:
panel_df["Ward Count"] = 0

panel_df.loc[panel_df["PSU Type"] == "ward", "Ward Count"] = 1
panel_df.loc[panel_df["PSU Type"] == "town_village", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "town_village"]
    .groupby("TownVillage")["UrbanWardVillage"]
    .transform("count")
)
panel_df.loc[panel_df["PSU Type"] == "subdistrict", "Ward Count"] = (
    panel_df[panel_df["PSU Type"] == "subdistrict"]
    .groupby("Subdistrict")["UrbanWardVillage"]
    .transform("count")
)
panel_df["Ward Count"] = panel_df["Ward Count"].astype(int)

#### Reorder columns

In [38]:
reordered_columns = [
    "State",
    "State_Name",
    "District",
    "District_Name",
    "Subdistrict",
    "Subd_Name",
    "TownVillage",
    "UrbanWardVillage",
    "WardVillage_Name",
    "PCA_ID",
    "TRU",
    "WardVillage_Pop",
    "Subd_Pop",
    "State_Pop",
    "WardVillageID",
    "Ward Boundary Available with MapSolve",
    "Included in Panel",
    "State Shared by MapSolve",
    "State Changed",
    "Ward Boundary Given",
    "TV Boundary Given",
    "SubDistrict Boundary Given",
    "Delivery State",
    "PSU Type",
    "Ward Count",
]
panel_df = panel_df[reordered_columns]

#### Save

In [39]:
panel_df.to_csv(CLEANED_DATA_DIR / "Panel Wards with Quality Checks.csv", index=False)

In [40]:
if not panel_df["State Shared by MapSolve"].all():
    print("There are states in the merged data that are not shared by MapSolve:")
    print(panel_df[~panel_df["State Shared by MapSolve"]]["State"].unique())
    print("Saving the states for which we DO have data separately:")
    panel_df[panel_df["State Shared by MapSolve"]].to_csv(
        CLEANED_DATA_DIR / "Panel Wards with Quality Checks - Shared States.csv",
        index=False,
    )

In [41]:
panel_df["Delivery State"].str.rfind("BAD")

0     -1
1     -1
2     -1
3     -1
4     -1
      ..
707   -1
708   -1
709   -1
710   -1
711   -1
Name: Delivery State, Length: 712, dtype: int64

#### Save per-state stats

In [40]:
# Get counts of delivery states by state
delivery_state_counts = (
    panel_df[panel_df["State"].isin(given_states_list)]
    .groupby("State")["Delivery State"]
    .value_counts()
)

# Convert to a more readable DataFrame format
delivery_state_pivot = delivery_state_counts.unstack(fill_value=0).reset_index()

# Sort by state code
delivery_state_pivot = delivery_state_pivot.sort_values(by="State")

# Add state names for better readability
state_name_mapping = (
    panel_df[["State", "State_Name"]]
    .drop_duplicates()
    .set_index("State")["State_Name"]
)
delivery_state_pivot["State_Name"] = delivery_state_pivot["State"].map(
    state_name_mapping
)

# Reorder columns to show State_Name first
columns = ["State", "State_Name"] + [
    col for col in delivery_state_pivot.columns if col not in ["State", "State_Name"]
]
delivery_state_pivot = delivery_state_pivot[columns]

# transform the DataFrame to have the delivery states as rows and state names as columns
delivery_state_pivot.drop(columns=["State"], inplace=True)
delivery_state_pivot = delivery_state_pivot.set_index("State_Name").T.reset_index()
delivery_state_pivot

State_Name,Delivery State,HIMACHAL PRADESH,PUNJAB,UTTARAKHAND,HARYANA,NCT OF DELHI,RAJASTHAN,UTTAR PRADESH,BIHAR,TRIPURA,...,ODISHA,CHHATTISGARH,MADHYA PRADESH,GUJARAT,MAHARASHTRA,ANDHRA PRADESH,KARNATAKA,KERALA,TAMIL NADU,TELANGANA
0,BAD - No boundary(s) given,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1,BAD - Town/Village boundary given but Ward bou...,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,BETTER - Ward boundary given but only TV or Su...,3,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,2,0
3,GOOD - Subdistrict boundary given as expected,0,0,31,0,28,0,0,0,4,...,0,1,0,0,0,0,0,0,0,0
4,GOOD - Town/Village boundary given as expected,0,11,0,12,0,1,36,31,20,...,9,8,5,8,16,24,4,28,22,12
5,GOOD - Ward boundary given as expected,0,18,2,10,73,8,12,7,0,...,9,10,12,32,21,0,29,4,17,18


In [41]:
# add a total column as the second column
delivery_state_pivot.insert(1, "Total", delivery_state_pivot.iloc[:, 1:].sum(axis=1))

In [42]:
delivery_state_pivot

State_Name,Delivery State,Total,HIMACHAL PRADESH,PUNJAB,UTTARAKHAND,HARYANA,NCT OF DELHI,RAJASTHAN,UTTAR PRADESH,BIHAR,...,ODISHA,CHHATTISGARH,MADHYA PRADESH,GUJARAT,MAHARASHTRA,ANDHRA PRADESH,KARNATAKA,KERALA,TAMIL NADU,TELANGANA
0,BAD - No boundary(s) given,2,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1,BAD - Town/Village boundary given but Ward bou...,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,BETTER - Ward boundary given but only TV or Su...,7,3,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,2,0
3,GOOD - Subdistrict boundary given as expected,64,0,0,31,0,28,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,GOOD - Town/Village boundary given as expected,324,0,11,0,12,0,1,36,31,...,9,8,5,8,16,24,4,28,22,12
5,GOOD - Ward boundary given as expected,314,0,18,2,10,73,8,12,7,...,9,10,12,32,21,0,29,4,17,18


In [43]:
delivery_state_pivot.to_csv(
    CLEANED_DATA_DIR / "Delivery State Counts By State.csv", index=False
)

#### Old

In [44]:
# unique_requested_tv_codes = set(
#     panel_df[panel_df["State Shared by MapSolve"]]["TownVillage"]
# )
# unique_received_tv_codes = set(gdf.loc[~gdf["TV_C"].isna(), "TV_C"].astype(int))
# matched_tv_codes = unique_received_tv_codes.intersection(unique_requested_tv_codes)

In [45]:
# print("Number of requested TV codes:", len(unique_requested_tv_codes))
# print("Number of received TV codes:", len(unique_received_tv_codes))

# print("Number of matched TV codes:", len(matched_tv_codes))
# print(
#     "Number of TV codes not received:",
#     len(unique_requested_tv_codes.difference(matched_tv_codes)),
# )

In [46]:
# # show the received breakdown by state
# received_pivot_table = (
#     panel_df[panel_df["State Shared by MapSolve"]]
#     .groupby(
#         [
#             "State",
#             "State_Name",
#             "District",
#             "District_Name",
#             "Subdistrict",
#             "Subd_Name",
#         ]
#     )["Delivery State"]
#     .value_counts()
#     .unstack(fill_value=0)
# ).reset_index()

# # received_pivot_table.rename(
# #     columns={True: "TV_Codes Received", False: "TV_Codes Not Received"}, inplace=True
# # )

In [47]:
# received_pivot_table

In [48]:
# pivot_table_not_received = received_pivot_table[
#     received_pivot_table["BAD - No boundary(s) given"] > 0
# ]
# pivot_table_not_received


In [49]:
# pivot_table_not_received.to_csv(
#     CLEANED_DATA_DIR / "Requested TV Codes Not Received Subdistrict Breakdown.csv",
#     index=False,
# )

In [50]:
# requested_tv_codes_not_received = panel_df[
#     (panel_df["State Shared by MapSolve"])
#     & (panel_df["Delivery State"] == "BAD - No boundary(s) given")
# ]
# requested_tv_codes_not_received

In [51]:
# requested_tv_codes_not_received.to_csv(
#     CLEANED_DATA_DIR / "Requested TV Codes Not Received.csv", index=False
# )

## Post-sampling checks

In [52]:
# check out district Chittoor, tv_code: 803014. Seems like we're missing the subdistrict name and ID is changed.

# ignore: (NCT of) Delhi, Tripura, Meghalaya

In [53]:
# OUTPUT_DATA_DIR = DATA_DIR / "03. Final Outputs"

In [54]:
# v4 = gpd.read_parquet(
#     OUTPUT_DATA_DIR
#     / "v4"
#     / "01. Sampled Rooftop Data"
#     / "sampled_rooftops_snapped_points.parquet"
# )
# v4["source_version"] = "v4"

# v6_uttarakhand = gpd.read_parquet(
#     OUTPUT_DATA_DIR
#     / "v6_uttarakhand"
#     / "01. Sampled Rooftop Data"
#     / "Uttarakhand"
#     / "Uttarakhand_sampled_rooftops_snapped_points.parquet"
# )
# v6_uttarakhand["source_version"] = "v6_uttarakhand"

# v7_leftouts = gpd.read_parquet(
#     OUTPUT_DATA_DIR
#     / "v7_leftouts"
#     / "01. Sampled Rooftop Data"
#     / "sampled_rooftops_snapped_points.parquet"
# )
# v7_leftouts["source_version"] = "v7_leftouts"

In [55]:
# combined_rooftops_gdf = gpd.GeoDataFrame(
#     pd.concat([v4, v6_uttarakhand, v7_leftouts], ignore_index=True)
# )

In [56]:
# combined_rooftops_gdf.columns

In [57]:
# duplicated_gdf = combined_rooftops_gdf[
#     combined_rooftops_gdf["Rooftop Unique ID"].duplicated(keep=False)
# ].sort_values("Rooftop Unique ID")

In [58]:
# # load Red's DQ checks
# missing_ward_df = pd.read_excel(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_v4.xlsx",
#     sheet_name="ward_unmerged",
# )
# missing_tv_df = pd.read_excel(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_v4.xlsx",
#     sheet_name="tv_unmerged",
# )
# missing_subd_df = pd.read_excel(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_v4.xlsx",
#     sheet_name="subdistrict_unmerged",
# )

In [59]:
# missing_ward_df["psutype"] = missing_ward_df["deliverystate"].map(psu_mapping)
# missing_tv_df["psutype"] = missing_tv_df["deliverystate"].map(psu_mapping)
# missing_subd_df["psutype"] = missing_subd_df["deliverystate"].map(psu_mapping)

### Ward

In [60]:
# missing_ward_df.groupby("deliverystate").size()

In [61]:
# # add a new column that says "PSU Type in Rooftop Sample" that is true if found, False if not
# missing_ward_df["PSU Type in Rooftop Sample"] = False

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "ward",
#     "PSU Type in Rooftop Sample",
# ] = missing_ward_df["pca_id"].isin(combined_rooftops_gdf["PCA_ID"].unique())

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "town_village",
#     "PSU Type in Rooftop Sample",
# ] = missing_ward_df["townvillage"].isin(
#     combined_rooftops_gdf[
#         combined_rooftops_gdf["Ward Code"].isnull()
#         & combined_rooftops_gdf["TV Code"].notna()
#     ]["TV Code"].unique()
# )

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "subdistrict",
#     "PSU Type in Rooftop Sample",
# ] = missing_ward_df["subdistrict"].isin(
#     combined_rooftops_gdf[
#         (combined_rooftops_gdf["Ward Code"].isnull())
#         & (combined_rooftops_gdf["TV Code"].isnull())
#         & (combined_rooftops_gdf["Subdistrict Code"].notna())
#     ]["Subdistrict Code"].unique()
# )

# # add a new column that says "PSU Type in MapSolve Shapes" that is based on the "gdf" dataset that is true if found, False if not
# missing_ward_df["PSU Type in MapSolve Shapes"] = False

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "ward",
#     "PSU Type in MapSolve Shapes",
# ] = missing_ward_df["pca_id"].isin(gdf["PCA_ID"].unique())

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "town_village",
#     "PSU Type in MapSolve Shapes",
# ] = missing_ward_df["townvillage"].isin(
#     gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()]["TV_C"].unique()
# )

# missing_ward_df.loc[
#     missing_ward_df["psutype"] == "subdistrict",
#     "PSU Type in MapSolve Shapes",
# ] = missing_ward_df["subdistrict"].isin(
#     gdf[(gdf["Ward_C"].isnull()) & (gdf["TV_C"].isnull()) & (gdf["SubDist_C"].notna())][
#         "SubDist_C"
#     ].unique()
# )

In [62]:
# missing_ward_df

In [63]:
# missing_ward_df["PSU Type in Rooftop Sample"].value_counts()

### TV

In [64]:
# missing_tv_df.groupby("deliverystate").size()

In [65]:
# # add a new column that says "PSU Type in Rooftop Sample" that is true if found, False if not
# missing_tv_df["PSU Type in Rooftop Sample"] = False

# missing_tv_df.loc[
#     missing_tv_df["psutype"].isin(["ward"]),
#     "PSU Type in Rooftop Sample",
# ] = missing_tv_df["pca_id"].isin(combined_rooftops_gdf["PCA_ID"].unique())

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "town_village",
#     "PSU Type in Rooftop Sample",
# ] = missing_tv_df["tv_id"].isin(
#     combined_rooftops_gdf[
#         combined_rooftops_gdf["Ward Code"].isnull()
#         & combined_rooftops_gdf["TV Code"].notna()
#     ]["TV Code"].unique()
# )

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "subdistrict",
#     "PSU Type in Rooftop Sample",
# ] = missing_tv_df["subdistrict"].isin(
#     combined_rooftops_gdf[
#         (combined_rooftops_gdf["Ward Code"].isnull())
#         & (combined_rooftops_gdf["TV Code"].isnull())
#         & (combined_rooftops_gdf["Subdistrict Code"].notna())
#     ]["Subdistrict Code"].unique()
# )

# # add a new column that says "PSU Type in MapSolve Shapes" that is based on the "gdf" dataset that is true if found, False if not
# missing_tv_df["PSU Type in MapSolve Shapes"] = False

# missing_tv_df.loc[
#     missing_tv_df["psutype"].isin(["ward"]),
#     "PSU Type in MapSolve Shapes",
# ] = missing_tv_df["pca_id"].isin(gdf["PCA_ID"].unique())

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "town_village",
#     "PSU Type in MapSolve Shapes",
# ] = missing_tv_df["tv_id"].isin(
#     gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()]["TV_C"].unique()
# )

# missing_tv_df.loc[
#     missing_tv_df["psutype"] == "subdistrict",
#     "PSU Type in MapSolve Shapes",
# ] = missing_tv_df["subdistrict"].isin(
#     gdf[(gdf["Ward_C"].isnull()) & (gdf["TV_C"].isnull()) & (gdf["SubDist_C"].notna())][
#         "SubDist_C"
#     ].unique()
# )

In [66]:
# missing_tv_df

In [67]:
# missing_tv_df["PSU Type in Rooftop Sample"].value_counts()

### Subdistrict

In [68]:
# missing_subd_df.groupby("deliverystate").size()

In [69]:
# # add a new column that says "PSU Type in Rooftop Sample" that is true if found, False if not
# missing_subd_df["PSU Type in Rooftop Sample"] = False

# missing_subd_df.loc[
#     missing_subd_df["psutype"].isin(["ward"]),
#     "PSU Type in Rooftop Sample",
# ] = missing_subd_df["pca_id"].isin(combined_rooftops_gdf["PCA_ID"].unique())

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "town_village",
#     "PSU Type in Rooftop Sample",
# ] = missing_subd_df["tv_id"].isin(
#     combined_rooftops_gdf[
#         combined_rooftops_gdf["Ward Code"].isnull()
#         & combined_rooftops_gdf["TV Code"].notna()
#     ]["TV Code"].unique()
# )

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "subdistrict",
#     "PSU Type in Rooftop Sample",
# ] = missing_subd_df["subdistrict_census_id"].isin(
#     combined_rooftops_gdf[
#         (combined_rooftops_gdf["Ward Code"].isnull())
#         & (combined_rooftops_gdf["TV Code"].isnull())
#         & (combined_rooftops_gdf["Subdistrict Code"].notna())
#     ]["Subdistrict Code"].unique()
# )

# # add a new column that says "PSU Type in MapSolve Shapes" that is based on the "gdf" dataset that is true if found, False if not
# missing_subd_df["PSU Type in MapSolve Shapes"] = False

# missing_subd_df.loc[
#     missing_subd_df["psutype"].isin(["ward"]),
#     "PSU Type in MapSolve Shapes",
# ] = missing_subd_df["pca_id"].isin(gdf["PCA_ID"].unique())

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "town_village",
#     "PSU Type in MapSolve Shapes",
# ] = missing_subd_df["townvillage"].isin(
#     gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()]["TV_C"].unique()
# )

# missing_subd_df.loc[
#     missing_subd_df["psutype"] == "subdistrict",
#     "PSU Type in MapSolve Shapes",
# ] = missing_subd_df["subdistrict_census_id"].isin(
#     gdf[(gdf["Ward_C"].isnull()) & (gdf["TV_C"].isnull()) & (gdf["SubDist_C"].notna())][
#         "SubDist_C"
#     ].unique()
# )

In [70]:
# missing_subd_df["PSU Type in Rooftop Sample"].value_counts()

In [71]:
# # recreate excel and save to file
# with pd.ExcelWriter(
#     CLEANED_DATA_DIR.parent.parent / "red_sample_dq_check_updated_v4.xlsx"
# ) as writer:
#     missing_ward_df.to_excel(writer, sheet_name="ward_unmerged", index=False)
#     missing_tv_df.to_excel(writer, sheet_name="tv_unmerged", index=False)
#     missing_subd_df.to_excel(writer, sheet_name="subdistrict_unmerged", index=False)


### Get the shapes that were missed

In [72]:
# missing_tv_df.rename(columns={"tv_id": "TV Code"}, inplace=True)
# missing_ward_df.rename(columns={"townvillage": "TV Code"}, inplace=True)
# missing_subd_df.rename(columns={"TownVillage": "TV Code"}, inplace=True)

In [73]:
# combined_missing_df = pd.concat(
#     [missing_ward_df, missing_tv_df, missing_subd_df],
#     ignore_index=True,
# )

In [74]:
# combined_missing_df.drop(columns=["townvillage", "tv_id", "TownVillage"], inplace=True)

In [75]:
# unavailable_shapes_df = combined_missing_df[~combined_missing_df["PSU Type in MapSolve Shapes"]]
# unavailable_shapes_df

In [76]:
# unavailable_shapes_df.to_csv(
#     CLEANED_DATA_DIR / "Unavailable Shapes in MapSolve.csv",
#     index=False,
# )

In [77]:
# # filter out the rows where "PSU Type in Rooftop Sample" is True
# combined_missing_df = combined_missing_df[
#     ~combined_missing_df["PSU Type in Rooftop Sample"] &
#     combined_missing_df["PSU Type in MapSolve Shapes"]
# ]
# combined_missing_df

In [78]:
# # drop any rows in states DELHI, NCT OF DELHI, TRIPURA, MEGHALAYA
# combined_missing_df = combined_missing_df[
#     ~combined_missing_df["State_Name"].isin(
#         ["DELHI", "NCT OF DELHI", "TRIPURA", "MEGHALAYA"]
#     )
# ]
# combined_missing_df

In [79]:
# # deduplicate to avoid merging issues
# ward_missing_df = combined_missing_df[
#     combined_missing_df["psutype"] == "ward"
# ].drop_duplicates(subset=["pca_id"])
# tv_missing_df = combined_missing_df[
#     combined_missing_df["psutype"] == "town_village"
# ].drop_duplicates(subset=["TV Code"])
# subd_missing_df = combined_missing_df[
#     combined_missing_df["psutype"] == "subdistrict"
# ].drop_duplicates(subset=["subd_name"])

In [80]:
# combined_missing_df[combined_missing_df["deliverystate"] == "BAD - Subdistrict boundary given but Ward boundaries expected"].to_csv(
#     CLEANED_DATA_DIR / "Ward Expected but Subdistrict Given.csv",
#     index=False,
# )

In [81]:
# pd.concat(
#     [ward_missing_df, tv_missing_df, subd_missing_df],
#     ignore_index=True,
# )

In [82]:
# # Ward shapes
# leftout_wards_gdf = gdf.merge(
#     ward_missing_df,
#     left_on="PCA_ID",
#     right_on="pca_id",
#     how="inner",
#     suffixes=("", "_missing"),
# )

# # TV shapes
# leftout_tvs_gdf = gdf[gdf["Ward_C"].isnull() & gdf["TV_C"].notna()].merge(
#     tv_missing_df,
#     left_on="TV_C",
#     right_on="TV Code",
#     how="inner",
#     suffixes=("", "_missing"),
# )

# # Subdistrict shapes
# leftout_subd_gdf = gdf[
#     gdf["Ward_C"].isnull() & gdf["TV_C"].isnull() & gdf["SubDist_C"].notna()
# ].merge(
#     subd_missing_df,
#     left_on="SubDist_N",
#     right_on="subd_name",
#     how="inner",
#     suffixes=("", "_missing"),
# )

In [83]:
# all_leftout_gdf_merge = pd.concat(
#     [leftout_wards_gdf, leftout_tvs_gdf, leftout_subd_gdf],
#     ignore_index=True,
# )

In [84]:
# all_leftout_gdf_merge.duplicated(keep=False).sum()

In [85]:
# all_leftout_gdf_merge

In [86]:
# all_leftout_gdf_merge.plot()

In [87]:
# all_leftout_gdf_merge = all_leftout_gdf_merge[
#     [
#         "UID",
#         "PCA_ID",
#         "State_C",
#         "State_N",
#         "Dist_C",
#         "Dist_N",
#         "SubDist_C",
#         "SubDist_N",
#         "TV_C",
#         "TV_N",
#         "Ward_C",
#         "TOT_P",
#         "geometry",
#         "state",
#         "state_name",
#         "district",
#         "district_name",
#         "subdistrict",
#         "subd_name",
#         # "townvillage",
#         "TV Code",
#         "ward_id",
#         "wardvillage_name",
#         "pca_id",
#         "tru",
#         "wardvillage_pop",
#         "subd_pop",
#         "state_pop",
#         "wardvillageid",
#         "wardboundaryavailablewithmapsolv",
#         "sourcesheet",
#         "sampledforpanel",
#         "sampledforpanelround2",
#         "sampledforifs",
#         "statesharedbymapsolve",
#         "statechanged",
#         "wardboundarygiven",
#         "tvboundarygiven",
#         "subdistrictboundarygiven",
#         "deliverystate",
#         "psutype",
#         "wardcountifs",
#         "wardcountpanel",
#         "wardcountpanelround2",
#         "wardcountall",
#         "state_census_id",
#         "state_census_name",
#         "district_census_id",
#         "district_census_name",
#         "subdistrict_census_id",
#         "subdistrict_census_name",
#         # "tv_id",
#         "tv_name",
#         "target_psu",
#         "target_psu_type",
#         "targets_count",
#         "target_pin_type",
#         "pin_id",
#         "pin_name",
#         "gadm_state_name",
#         "gadm_district_name",
#         "gadm_subdistrict_name",
#         "google_maps",
#         "latitude",
#         "longitude",
#         "target_id",
#         "my_maps",
#         "State",
#         "State_Name",
#         "District",
#         "District_Name",
#         "Subdistrict",
#         "Subd_Name",
#         # "TownVillage",
#         "WardVillage_Name",
#         "TRU",
#         "WardVillage_Pop",
#         "Subd_Pop",
#         "State_Pop",
#         "WardVillageID",
#         "WardBoundaryAvailablewithMap",
#         "_m2",
#         "_merge",
#         "PSU Type in Rooftop Sample",
#         "PSU Type in MapSolve Shapes",
#         "urbanwardvillage",
#         "UrbanWardVillage",
#         "tv_ward_count",
#         "tv_sample_count",
#         "subdist_ward_count",
#         "subdist_sample_count",
#         "_m3",
#     ]
# ]

In [88]:
# all_leftout_gdf_merge["UID"] = all_leftout_gdf_merge["UID"].astype(float)

In [89]:
# save_shapefiles(
#     all_leftout_gdf_merge,
#     CLEANED_DATA_DIR / "Leftout Shapes",
#     "leftout_shapes_after_v4_v6_uttarakhand_v7_leftouts",
#     formats=["csv", "parquet", "kml"],
# )